# 05 — Backtest Validation (Equities: SHORT/FLAT/LONG)
Runs a deterministic bar-by-bar backtest of a trained **DeepScalper** model on the
held-out **test split** (last 20%, matching 70/10/20 split used in training).

**How it works:**
- Loads `{SYMBOL}.parquet` and recomputes features via the shared pipeline.
- Uses time-ordered split: 70% train / 10% val / 20% test.
- Builds Dict observations `{lob, priv, macro}` and runs **greedy** (ε=0) inference.
- Uses 3-action direction branch: `0=SHORT`, `1=FLAT`, `2=LONG`.
- Tracks portfolio value at 1-minute resolution with paper-like transaction cost.

**Output:**
- Metrics table: total return, annualised Sharpe (equity session), max drawdown, win rate, trade count.
- Equity curve plot for selected symbol.

**Input:**  `/content/drive/MyDrive/algo_trader/weights/{SYMBOL}.pth`
           `/content/drive/MyDrive/algo_trader/data/raw/{SYMBOL}.parquet`

In [ ]:
!pip install -q torch pyarrow pandas numpy matplotlib tqdm pytz

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

WEIGHTS_DIR = '/content/drive/MyDrive/algo_trader/weights'
RAW_DIR     = '/content/drive/MyDrive/algo_trader/data/raw'
REPO_DIR    = '/content/deepscalper_copilot'

print(f'Weights dir : {WEIGHTS_DIR}')
print(f'Raw data dir: {RAW_DIR}')

In [ ]:
REPO_URL = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'

if not os.path.exists(REPO_DIR + '/.git'):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# algo_trader on sys.path → enables `from colab.deepscalper.X import Y`
ALGO_DIR = REPO_DIR + '/algo_trader'
if ALGO_DIR not in sys.path:
    sys.path.insert(0, ALGO_DIR)

print('Repo on path ✓')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from pathlib import Path

from colab.deepscalper.architecture import DeepScalperNet
from colab.deepscalper.utils import (
    compute_macro_features,
    compute_micro_features,
    compute_day_starts,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

# Current architecture config
MACRO_DIM     = 11
LOB_DIM       = 5
PRIV_DIM      = 2
N_DIR         = 3
N_SIZE        = 4
GRU_HIDDEN    = 128
MACRO_EMBED   = 64
FC_HIDDEN     = 128
LOOKBACK_BARS = 60

# Data split and transaction cost
a = 0.70  # train
b = 0.10  # val
# test = 0.20
TC_PCT = 0.0001
ANNUALISE = np.sqrt(98_280)  # US equity regular-session minute bars

SYMBOL = 'AAPL'
SAFE_NAME = SYMBOL.replace('/', '_')
print(f'Backtesting symbol: {SYMBOL}')

In [ ]:
# Backtest engine (3-action SHORT/FLAT/LONG)

def run_backtest(
    model: DeepScalperNet,
    macro_feats: np.ndarray,
    lob_feats: np.ndarray,
    close_arr: np.ndarray,
    day_starts: list,
    lookback: int = 60,
    tc_pct: float = 0.0001,
    device: str = 'cpu',
) -> dict:
    model.eval()

    portfolio = 1.0
    equity_curve = [1.0]
    step_returns = []
    trade_pnls = []

    n_bars = len(close_arr)

    for day_i, day_start in enumerate(day_starts):
        day_end = day_starts[day_i + 1] - 1 if day_i + 1 < len(day_starts) else n_bars - 1
        if day_start + lookback >= day_end:
            continue

        position = 0          # -1 short, 0 flat, +1 long
        entry_price = 0.0
        priv_history = deque([np.zeros(2, dtype=np.float32)] * lookback, maxlen=lookback)

        for t in range(day_start + lookback - 1, day_end):
            win_start = max(0, t - lookback + 1)

            lob_seq = lob_feats[win_start:t + 1]
            if len(lob_seq) < lookback:
                pad = np.zeros((lookback - len(lob_seq), lob_feats.shape[1]), dtype=np.float32)
                lob_seq = np.vstack([pad, lob_seq])

            priv_seq = np.array(list(priv_history), dtype=np.float32)
            macro = macro_feats[t]

            with torch.no_grad():
                lob_t = torch.tensor(lob_seq[None], dtype=torch.float32, device=device)
                prv_t = torch.tensor(priv_seq[None], dtype=torch.float32, device=device)
                mac_t = torch.tensor(macro[None], dtype=torch.float32, device=device)
                q_dir, _, _ = model(lob_t, prv_t, mac_t)

            action = int(q_dir.argmax(1).item())  # 0=SHORT, 1=FLAT, 2=LONG
            if action == 0:
                new_position = -1
            elif action == 1:
                new_position = 0
            else:
                new_position = 1

            current_price = float(close_arr[t])
            next_price = float(close_arr[min(t + 1, day_end)])
            tc_cost = 0.0

            # Realize PnL when closing/reversing an existing position.
            if position == 1 and new_position != 1 and entry_price > 0:
                pnl = (current_price - entry_price) / (entry_price + 1e-10)
                trade_pnls.append(pnl)
            elif position == -1 and new_position != -1 and entry_price > 0:
                pnl = (entry_price - current_price) / (entry_price + 1e-10)
                trade_pnls.append(pnl)

            turnover = abs(new_position - position)
            if turnover > 0:
                tc_cost += tc_pct * turnover
                entry_price = current_price if new_position != 0 else 0.0

            base_log_ret = np.log(next_price / (current_price + 1e-10)) if current_price > 0 else 0.0
            if position == 1:
                step_ret = base_log_ret
            elif position == -1:
                step_ret = -base_log_ret
            else:
                step_ret = 0.0
            step_r = step_ret - tc_cost

            position = new_position
            step_returns.append(step_r)
            portfolio *= float(np.exp(step_r))
            equity_curve.append(portfolio)

            if position == 1 and entry_price > 0:
                unreal_pnl = (current_price - entry_price) / (entry_price + 1e-10)
            elif position == -1 and entry_price > 0:
                unreal_pnl = (entry_price - current_price) / (entry_price + 1e-10)
            else:
                unreal_pnl = 0.0
            priv_history.append(np.array([float(position), float(np.clip(unreal_pnl, -0.5, 0.5))], dtype=np.float32))

        # Force close at end of day
        if position == 1 and entry_price > 0:
            eod_price = float(close_arr[day_end])
            pnl = (eod_price - entry_price) / (entry_price + 1e-10)
            trade_pnls.append(pnl)
        elif position == -1 and entry_price > 0:
            eod_price = float(close_arr[day_end])
            pnl = (entry_price - eod_price) / (entry_price + 1e-10)
            trade_pnls.append(pnl)

    arr = np.array(step_returns, dtype=np.float64)
    sharpe = float((arr.mean() / (arr.std() + 1e-10)) * ANNUALISE) if len(arr) > 1 else 0.0

    eq = np.array(equity_curve, dtype=np.float64)
    peak = np.maximum.accumulate(eq)
    max_dd = float(((eq - peak) / (peak + 1e-10)).min())

    return {
        'equity_curve': equity_curve,
        'total_return': float(portfolio - 1.0),
        'sharpe': sharpe,
        'max_drawdown': max_dd,
        'win_rate': float(np.mean([p > 0 for p in trade_pnls])) if trade_pnls else 0.0,
        'n_trades': len(trade_pnls),
    }


# Load data
wt_path = Path(WEIGHTS_DIR) / f'{SAFE_NAME}.pth'
raw_path = Path(RAW_DIR) / f'{SAFE_NAME}.parquet'

if not wt_path.exists():
    raise FileNotFoundError(f'Missing weights file: {wt_path}')
if not raw_path.exists():
    raise FileNotFoundError(f'Missing raw data file: {raw_path}')

bars = pd.read_parquet(str(raw_path))
bars.columns = [c.lower() for c in bars.columns]
bars = bars[['open', 'high', 'low', 'close', 'volume']].astype(float)

macro_feats = compute_macro_features(bars)
lob_feats = compute_micro_features(bars, use_proxy=True)
close_arr = bars['close'].values.astype(np.float64)
day_starts = compute_day_starts(bars.index)

n_bars = len(bars)
train_end = int(n_bars * 0.70)
val_end = int(n_bars * 0.80)

test_day_starts = [d - val_end for d in day_starts if d >= val_end]
if len(test_day_starts) < 1:
    raise RuntimeError('Insufficient test days after 70/10/20 split.')

# Load model checkpoint
ckpt = torch.load(str(wt_path), map_location=DEVICE, weights_only=True)
state_dict = ckpt.get('online_net', ckpt)

model = DeepScalperNet(
    macro_dim=MACRO_DIM,
    lob_dim=LOB_DIM,
    priv_dim=PRIV_DIM,
    gru_hidden=GRU_HIDDEN,
    macro_embed=MACRO_EMBED,
    fc_hidden=FC_HIDDEN,
    n_dir=N_DIR,
    n_size=N_SIZE,
)
model.load_state_dict(state_dict)
model = model.to(DEVICE)

result = run_backtest(
    model=model,
    macro_feats=macro_feats[val_end:],
    lob_feats=lob_feats[val_end:],
    close_arr=close_arr[val_end:],
    day_starts=test_day_starts,
    lookback=LOOKBACK_BARS,
    tc_pct=TC_PCT,
    device=DEVICE,
)

print(
    f"{SYMBOL}  Return={result['total_return']*100:+7.2f}%  "
    f"Sharpe={result['sharpe']:+6.3f}  "
    f"MaxDD={result['max_drawdown']*100:6.2f}%  "
    f"WinRate={result['win_rate']*100:5.1f}%  "
    f"Trades={result['n_trades']}"
)

In [ ]:
# Backtest report + equity curve
print('=' * 70)
print('  DEEPSCALPER BACKTEST REPORT (equity test split)')
print('=' * 70)
print(f"  Symbol          : {SYMBOL}")
print(f"  Total Return    : {result['total_return']*100:+.2f}%")
print(f"  Sharpe (annual) : {result['sharpe']:+.3f}")
print(f"  Max Drawdown    : {result['max_drawdown']*100:.2f}%")
print(f"  Win Rate        : {result['win_rate']*100:.1f}%")
print(f"  Trades          : {result['n_trades']}")
print('=' * 70)

curve = np.array(result['equity_curve'], dtype=np.float64)

fig, ax = plt.subplots(figsize=(14, 5), facecolor='#0d1117')
ax.set_facecolor('#0d1117')
ax.plot(curve, color='#00d4aa', linewidth=1.8, label=f'{SYMBOL} equity curve')
ax.axhline(1.0, color='#8b949e', linestyle='--', linewidth=0.8, label='Breakeven (1.0x)')

ax.set_title(f'DeepScalper — {SYMBOL} Test Equity Curve (1-min bars)', color='white', fontsize=13)
ax.set_xlabel('Bar step', color='#8b949e')
ax.set_ylabel('Normalised portfolio value', color='#8b949e')
ax.tick_params(colors='#8b949e')
ax.legend(facecolor='#161b22', labelcolor='white', fontsize=9)
for spine in ax.spines.values():
    spine.set_color('#30363d')

plt.tight_layout()
out_path = f'/content/{SAFE_NAME.lower()}_equity_curve.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Equity curve saved -> {out_path}')